# AdaptiveHb — Real Experiment (PyTorch backbones)

This is the notebook referenced by `PROJECT_MANIFEST.yaml`. It runs a genuine **baseline-vs-adaptive** experiment with the real PyTorch backbones on a dataset, then shows the archived metrics, the comparison (with paired significance), the reproducibility provenance manifest, and the publication figures.

**Requirements:** Python 3.11+, and the optional ML stack (PyTorch, torchvision, numpy, matplotlib, …). A GPU is recommended but not required.

Everything is driven from configuration — no dataset names, paths, image sizes, or model names are hardcoded.

## Get the code (Kaggle / Colab / local)

Run the cell below **first**. On Kaggle or Colab it clones the repository and enters it; when you already run from inside a local clone it does nothing.

In [ ]:
# === Get the code — run this FIRST (Kaggle / Colab / local) ==================
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/junaidmaqbool/AgenticHb.git"


def _has_repo(path) -> bool:
    return (Path(path) / "configs" / "project.yaml").is_file()


if _has_repo(Path.cwd()):
    pass                                        # already inside the repo (local run)
elif _has_repo(Path.cwd().parent):
    os.chdir(Path.cwd().parent)                 # notebook opened from notebooks/
elif Path("AgenticHb").exists() and _has_repo("AgenticHb"):
    # A previous clone exists — pull the latest so we never run stale code.
    subprocess.run(["git", "-C", "AgenticHb", "pull", "--ff-only"], check=False)
    os.chdir("AgenticHb")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "AgenticHb"], check=True)
    os.chdir("AgenticHb")

print("Working directory:", Path.cwd())
assert _has_repo(Path.cwd()), "Repository not found - check the clone step above."

# Sanity check: confirm we have the up-to-date code (real PNG synthetic images).
_syn = (Path.cwd() / "src" / "adaptivehb" / "dataset" / "synthetic.py").read_text()
assert "_png_bytes" in _syn, (
    "Stale code detected. Delete the old clone and re-run:  !rm -rf AgenticHb"
)
print("Code is up to date.")


## 0. Install the framework with the ML extras

Run this **once**. From a local clone (recommended), start the notebook from the repo root so the editable install resolves:

```bash
pip install -e ".[ml]"
```

On Google Colab, clone first, then install:

```python
!git clone https://github.com/<your-org>/AgenticHb.git
%cd AgenticHb
%pip install -e ".[ml]"
```

In [ ]:
# Uncomment to install (only needed once per environment).
# %pip install -e ".[ml]"

## 1. Setup and environment check

In [ ]:
# --- Make the framework importable and locate the repository root -------------
# Works whether you `pip install -e .` the package or just run from a clone.
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward from `start` (default: cwd) until a folder with configs/project.yaml."""
    start = (start or Path.cwd()).resolve()
    for directory in (start, *start.parents):
        if (directory / "configs" / "project.yaml").is_file():
            return directory
    raise FileNotFoundError(
        "Could not locate the repository root (a folder with configs/project.yaml). "
        "Run this notebook from inside the AgenticHb repository."
    )


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if SRC.is_dir() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))  # allows running without `pip install`

CONFIG_DIR = REPO_ROOT / "configs"
print("Repository root :", REPO_ROOT)
print("Config directory:", CONFIG_DIR)


In [ ]:
# Confirm the ML stack is present (this notebook needs it to train real backbones).
try:
    import torch
    print('torch            :', torch.__version__)
    print('CUDA available   :', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('CUDA device      :', torch.cuda.get_device_name(0))
except ImportError:
    print('PyTorch is NOT installed. Run the install cell in section 0 first.')
    print('(Without torch the pipeline still runs, but on reference models — use '
          'smoke_synthetic.ipynb for that.)')

## 2. Control panel — set your paths, tissues, and models here

Everything you need to change lives in the next cell. It loads the YAML config
and applies your overrides at runtime (the config files on disk are left
untouched).

**Expected dataset layout** (a tissue = a folder name; left/right are just
separate folders):

```
DATASET_ROOT/
├── images/
│   ├── left_eye/    P0001_left_eye.png, P0002_left_eye.png, ...
│   ├── right_eye/   P0001_right_eye.png, ...
│   └── palm/        P0001_palm.png, ...
├── masks/
│   ├── left_eye/    P0001_left_eye_mask.png, ...   (mask = <image>_mask.<ext>)
│   ├── right_eye/   ...
│   └── palm/        ...
└── metadata/
    └── patients.csv  (columns: Patient_ID, Hemoglobin, Age, Gender, ...)
```

Rules: the filename before the first `_` is the patient id (must match
`Patient_ID` in the CSV, no underscores in the id); masks are optional (a
warning if missing, but needed for segmentation).

In [ ]:
# ============================================================
# CONTROL PANEL — edit this cell, then run the cells below.
# ============================================================
from pathlib import Path
from adaptivehb.config import ConfigLoader
from adaptivehb.dataset import generate_synthetic_dataset

# --- Paths ---------------------------------------------------------------
DATASET_ROOT    = "/kaggle/input/your-dataset"   # your dataset root (layout above)
BASE_DIR        = str(REPO_ROOT / "runs")        # where ALL outputs are written
EXPERIMENT_NAME = "real_run"
USE_SYNTHETIC   = True    # True => ignore DATASET_ROOT and generate fake data to dry-run

# --- What to use (a list OVERRIDES the config; None keeps the config default) ---
TISSUES            = ["left_eye", "right_eye", "palm"]  # folder names under images/ & masks/
SEG_MODELS         = ["unet"]                           # segmentation architectures to train
PRED_BACKBONES     = ["efficientnet"]                   # prediction backbones allowed
PRED_DEFAULT       = "efficientnet"                     # default per-tissue backbone
PRED_TISSUE_MODELS = None            # e.g. {"left_eye": "vit"}; None = use default for all
METADATA_COLUMNS   = None            # e.g. ["Patient_ID", "Hemoglobin"] to relax required columns

# --- Training ------------------------------------------------------------
EPOCHS = 10                          # start small (3-5) to verify, then raise
# ============================================================

# Optionally generate a synthetic dataset (with your chosen tissue folders) to test.
if USE_SYNTHETIC:
    DATASET_ROOT = str(Path(BASE_DIR) / "synthetic_dataset")
    generate_synthetic_dataset(
        DATASET_ROOT, num_patients=24, seed=7,
        tissues=list(TISSUES) if TISSUES else None,
    )
    print("Generated a synthetic dataset at:", DATASET_ROOT)

# Load the YAML config, then apply the overrides above (source configs untouched).
config = ConfigLoader(CONFIG_DIR).load()
ds   = config.section("dataset")["dataset"]
seg  = config.section("segmentation")["segmentation"]
pred = config.section("prediction")["prediction"]

if TISSUES is not None:
    ds["tissues"] = list(TISSUES)
if METADATA_COLUMNS is not None:
    ds.setdefault("metadata", {})["mandatory_columns"] = list(METADATA_COLUMNS)
if SEG_MODELS is not None:
    seg["available_models"] = list(SEG_MODELS)
    seg["default_model"] = SEG_MODELS[0]
if PRED_BACKBONES is not None:
    pred["available_models"] = list(PRED_BACKBONES)
if PRED_DEFAULT is not None:
    pred["default_model"] = PRED_DEFAULT
if PRED_TISSUE_MODELS is not None:
    pred["tissue_models"] = dict(PRED_TISSUE_MODELS)

print("Dataset root :", DATASET_ROOT)
print("Output dir   :", BASE_DIR)
print("Tissues      :", ds["tissues"])
print("Segmentation :", seg["available_models"])
print("Prediction   :", pred["available_models"], "(default:", pred["default_model"] + ")")
print("Epochs       :", EPOCHS)


## 3. Run the experiment

This trains every segmentation and per-tissue prediction model, registers them with checkpoints, then compares the static baseline against the adaptive pipeline on the held-out test split and archives all outputs. Equivalent terminal command:

```bash
adaptivehb experiment --dataset-root <DATASET_ROOT> --base-dir runs --epochs 10 --name real_run
```

In [ ]:
from adaptivehb.pipeline import HbPipeline

# Build from the (overridden) config; base_dir/dataset_root come from the control panel.
pipeline = HbPipeline(config, base_dir=BASE_DIR, dataset_root=DATASET_ROOT)
result = pipeline.experiment(EXPERIMENT_NAME, epochs=EPOCHS)
print("Experiment id :", result.experiment_id)
print("Archived at   :", result.root)


## 4. Metrics (adaptive pipeline)

In [ ]:
import json

metrics = result.metrics
for key in ('mae', 'rmse', 'r2', 'pearson', 'spearman', 'mean_bias'):
    if key in metrics:
        print(f'{key:>10}: {metrics[key]:.4f}')

metrics_json = Path(result.root) / 'metrics' / 'adaptive_metrics.json'
print('\nFull metrics file:', metrics_json)

## 5. Baseline vs adaptive comparison (with paired significance)

In [ ]:
comparison = result.comparison
print(json.dumps({k: v for k, v in comparison.items() if k != 'significance'}, indent=2, default=str))

sig = comparison.get('significance')
if sig:
    print('\n-- paired significance --')
    print('n_pairs         :', sig['n_pairs'])
    print('mean |err| diff :', round(sig['mean_abs_error_diff'], 4))
    print('paired t-test p :', round(sig['paired_t_test']['p_value'], 6))
    print('wilcoxon p      :', round(sig['wilcoxon']['p_value'], 6))
    print("cohen's d       :", round(sig['cohens_d'], 4))
    ci = sig['bootstrap_ci']
    print('bootstrap 95% CI:', (round(ci['ci_lower'], 4), round(ci['ci_upper'], 4)))
    print('significant     :', sig['significant_at'])

## 6. Reproducibility provenance

In [ ]:
prov = result.provenance
print(json.dumps(prov, indent=2, default=str))

## 7. Publication figures

In [ ]:
from IPython.display import Image, display

figure_dir = Path(result.root) / 'figures'
pngs = sorted(figure_dir.glob('*.png')) if figure_dir.is_dir() else []
print('figures:', [p.name for p in pngs])
for png in pngs:
    display(Image(filename=str(png)))

## 8. Where everything is archived

In [ ]:
root = Path(result.root)
for sub in sorted(p for p in root.iterdir() if p.is_dir()):
    files = sorted(f.name for f in sub.rglob('*') if f.is_file())
    if files:
        print(f'{sub.name}/')
        for f in files:
            print('   ', f)

## Next steps

The archived experiment directory contains everything needed for the paper: the metric bundle, the baseline-vs-adaptive comparison with paired significance, the per-sample predictions, the figures (scatter, Bland-Altman, comparison), the reproducibility manifest, and a summary. Re-run with a larger `EPOCHS` and your real dataset to produce the final publication assets (Papers 1–4).